In [1]:
import bar_chart_race_._bar_chart_race_plotly as bcr

bcr = bcr._BarChartRace(#data_filename='FAOSTAT_data.csv',
                        #out_filename='race.html', 
                        period_length=500, 
                        interpolate_period=True, 
                        steps_per_period=1, n_bars=20, 
                        fixed_xaxis = True, #animation looks worse (or no animation) if False
                        bar_textposition='inside',
                        val_ax_label = '-log10(p.adj)',
                        scatter_values_inside_bar = True, # scatter inside the bar
                        #plot_pws_yaxis = True,
                        #linebreak_labels = True,
                        #labels_max_len = 100,
                        #linebreak_labels_len_greater = 80,
                        plot_labels_over_bars = True,
                        frame_subset=500)
bcr.make_animation()


creating frames: 100%|██████████| 500/500 [00:02<00:00, 233.54it/s]


In [ ]:
import pandas as pd
import numpy as np

steps_per_period = 1
n_bars = 10

df = pd.read_csv('./data/FAOSTAT_data.csv', index_col='Year')
df.sort_index(inplace=True)
items = df['Item'].unique()
item_idx_map = {name: i for i, name in enumerate(items)}
df["Item Index"] = df["Item"].map(item_idx_map)

df.drop(df.columns.difference(['Item Index','Value']), axis=1, inplace=True)

df_wide = pd.pivot_table(df, values='Value', index='Year', columns='Item Index')
df_wide = df_wide.fillna(0)
df_wide

df_wide_idx = df_wide.index
if df_wide.index[0] == 1:
    df_wide_idx -= 1

df_wide.index = df_wide_idx * steps_per_period
df_wide_idx = range(df_wide_idx[-1]+1)
df_wide = df_wide.reindex(df_wide_idx)

df_wide_interp = df_wide.interpolate()

# sort dataframe asc as matrix, grab n largest columns and reverse them
top_n = np.sort(df_wide_interp.to_numpy(), axis=1)[:, -n_bars:][:, ::-1]

df_vals = pd.DataFrame(top_n, index=df_wide_interp.index, columns=range(1, n_bars + 1))

# get a dataframe with label (pathway) indices ranked after their value in each window
df_ranks_wide = df_wide_interp.rank(axis=1, method='first', ascending=False)-1

df_ranks_wide[df_ranks_wide > n_bars-1] = np.nan
ser = df_ranks_wide.stack().reset_index()

df_ser = pd.DataFrame(ser).astype('int32')
df_ranks = df_ser.pivot(index=df_ser.columns[0], columns=0, values=df_ser.columns[1])
df_ranks



In [ ]:

import pandas as pd
import numpy as np

steps_per_period = 2
n_bars = 20

df_wide = pd.read_csv('./data/covid19.csv', index_col='date')
countries = df_wide.columns
df_wide.columns = [i for i in range(len(countries))]
#print(df_wide.head(1))
dates = df_wide.index
df_wide.index = range(len(dates))
df_wide_idx = df_wide.index

df_wide.index = df_wide_idx * steps_per_period
df_wide_idx = range(df_wide_idx[-1]+1)
df_wide = df_wide.reindex(df_wide_idx)

df_wide_interp = df_wide.interpolate()
#df_wide_interp = df_wide_interp.fillna(0)
print(df_wide_interp.head(2))
topN = np.sort(df_wide_interp.to_numpy(), axis=1)[:, -n_bars:][:, ::-1]
print('topN: ', topN)
df_vals = pd.DataFrame(topN, index=df_wide_interp.index, columns=range(1, n_bars + 1))
print('df vals')
print(df_vals.head(5))

df_ranks_wide = df_wide_interp.rank(axis=1, method='first', ascending=False)-1
df_ranks_wide[df_ranks_wide > n_bars-1] = np.nan
print('df_ranks_wide.head(2):')
print(df_ranks_wide.head(2))
ser = df_ranks_wide.stack().reset_index()
print('ranks ser:')
print(ser.head(10))
df_ser = pd.DataFrame(ser).astype('int32')
print('ranks df_ser:')
print(df_ser.head(20))
df_ranks = df_ser.pivot(index='level_0', columns=0, values=df_ser.columns[1])
df_ranks.head(20)